# 歌词分词，词性标注

In [31]:
import json
import pandas as pd


# import thulac


from collections import Counter
from openai import OpenAI

In [32]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [33]:
import sys
sys.path.append('..')

In [51]:
from songs.songs_libs import id_process

# 分词，词频与词性分析

In [34]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v',
    '春娇': 'n',
    '学会': 'v'
}

In [35]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [36]:
import re
from hanlp_restful import HanLPClient
HanLP = HanLPClient('https://www.hanlp.com/hanlp/v21/redirect', auth="699691e7eaf61a3aca90d7b8", language='zh')

def is_chinese_word(word):
    """
    判断是否为纯中文词
    """
    return 1 if re.fullmatch(r'[\u4e00-\u9fff]+', word) else 0


def process_lyrics_with_hanlp_multi_pos(text, word_to_fix=None):
    if not text:
        return []
    
    # 调用 HanLP
    result = HanLP.parse(text, tasks='pos/pku')
    
    sentences = result['tok/fine']
    pos_sentences = result['pos/pku']
    
    # 统计 (word, pos) -> freq
    word_pos_counter = Counter()
    
    for words, pos_tags in zip(sentences, pos_sentences):
        for word, tag in zip(words, pos_tags):
            
            word = word.strip()
            
            # 过滤标点
            if tag == 'w' or not word:
                continue
            
            # 词性修正
            if word_to_fix and word in word_to_fix:
                tag = word_to_fix[word]
            
            word_pos_counter[(word, tag)] += 1
    
    # 构建结果列表
    results = []
    for (word, pos), freq in word_pos_counter.items():
        results.append({
            "word": word,
            "pos": pos,
            "freq": freq,
            "is_chinese": is_chinese_word(word)
        })
    
    # 按词频排序
    results.sort(key=lambda x: x["freq"], reverse=True)
    
    return results


In [37]:
# thu = thulac.thulac(seg_only=False, filt=True) 

# def process_lyrics_with_thulac(text, word_to_fix=None):
#     if not text:
#         return []
    
#     # 2. 执行分词与词性标注
#     # 返回格式为 [[word, pos], [word, pos], ...]
#     words_with_pos = thu.cut(text)
    
#     # 3. 过滤无意义字符与词性修正
#     # thulac 的标点词性通常是 'w'
#     filtered_data = []
#     for word, pos in words_with_pos:
#         word = word.strip()
#         # 排除标点符号、空白字符
#         if pos != 'w' and len(word) > 0:
#             # 逻辑修正：word_to_fix 通常是修正词性
#             if word_to_fix and word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 4. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 5. 汇总信息
#     # 建立 word -> pos 映射
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count
#         })
    
#     return sorted_results

In [38]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'
    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            # lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
            #     i['lyrics_text'], word_to_fix=word_to_fix)
            lyric_words_dict[i['song_id']] = process_lyrics_with_hanlp_multi_pos(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    return df_word

In [39]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word = df_word.copy()
    df_songs = df_songs.copy()
    df_word['song_id'] = df_word['song_id'].astype(str)
    df_word['is_chinese'] = df_word['word'].apply(is_chinese_word)

    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# main

In [40]:
# file_path_prefix = "data/jaychou/"
# file_path_prefix = "data/mayday/"
# file_path_prefix = "data/liuyuning/"
# file_path_prefix = "data/newyear/"
file_path_prefix = "data/liyuchun/"

In [86]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year
0,641644435,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615,002ZOuVm3Qn20Y,清风醉,85081075,003yPfqX3jjXlP,235,1771466400,清风醉,清风醉,清风醉,2026-02-19,2026
1,625008,001DIsVq3sSjGV,下个，路口，见,NaN,李宇春,4615,002ZOuVm3Qn20Y,Chris Lee 同名专辑,53014,004Z4sId156J79,210,1261411200,下个，路口，见,Chris Lee 同名专辑,下个，路口，见,2009-12-22,2009
2,517385504,003SqJTl4fnhRC,大梦归离,《大梦归离》影视剧主题曲,李宇春,4615,002ZOuVm3Qn20Y,大梦归离 影视原声带,57721256,003up2WR1OtPci,281,1727233200,大梦归离,大梦归离 影视原声带,大梦归离,2024-09-25,2024
3,212992885,001TmXYt4B8W4d,蜀绣 (Live),NaN,李宇春,4615,002ZOuVm3Qn20Y,李宇春野蛮生长巡演LIVE自选,3899212,003IqTUf2cTV2s,307,1517328000,蜀绣,李宇春野蛮生长巡演LIVE自选,蜀绣,2018-01-31,2018
4,213468025,00467LeA1TSU69,1987我不知会遇见你,NaN,李宇春,4615,002ZOuVm3Qn20Y,李宇春2018流行（liú xíng）巡演LIVE辑,3975872,003UqEXI3NvbEl,332,1522512000,1987我不知会遇见你,李宇春2018流行（liú xíng）巡演LIVE辑,1987我不知会遇见你,2018-04-01,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,105206640,001oOToq0IUkiM,爱有引力,NaN,李宇春,4615,002ZOuVm3Qn20Y,混蛋，我想你,1235613,000wzySP2ujpo3,221,1449676800,爱有引力,混蛋，我想你,爱有引力,2015-12-10,2015
119,410614,0033rZ6R11iy99,我的黑白色,NaN,李宇春,4615,002ZOuVm3Qn20Y,我的,33189,001SqLCp2RidK7,301,1193932800,我的黑白色,我的,我的黑白色,2007-11-02,2007
120,426544,004JYRZc0OGTCi,TMD我爱你,NaN,李宇春,4615,002ZOuVm3Qn20Y,超级女声 美梦成真,34687,001uSTif44XUoj,200,1210780800,TMD我爱你,超级女声 美梦成真,TMD我爱你,2008-05-15,2008
121,225559362,003SszVH4IlNqh,木兰,《王者荣耀》花木兰英雄主打歌,李宇春,4615,002ZOuVm3Qn20Y,天美十年典藏：全明星音乐特辑,4867924,00436LYE43MyXL,201,1540483200,木兰,天美十年典藏：全明星音乐特辑,木兰,2018-10-26,2018


In [ ]:
# 五月天需要使用word_to_fix
# if file_path_prefix == "data/mayday/":
#     df_word = lyric_words_process(file_path_prefix, word_to_fix)
# else:
#     df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [42]:
# hanlp暂时不需要word_to_fix
df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
df_word

,song_id,word,pos,freq
0,641644435,我,r,16
1,641644435,清风,n,7
2,641644435,天地,n,5
3,641644435,伴,v,5
4,641644435,醉,v,5
...,...,...,...,...
11797,475987700,天赋,n,1
11798,475987700,惊喜,a,1
11799,475987700,哭泣,v,1
11800,475987700,儿时,t,1


In [87]:
df_merged = words_data_merge(df_word, df_songs)
df_merged = df_merged.dropna(subset='song_name')
df_merged

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,album_name,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year
0,641644435,我,r,16,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
1,641644435,清风,n,7,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
2,641644435,天地,n,5,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
3,641644435,伴,v,5,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
4,641644435,醉,v,5,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11797,475987700,天赋,n,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,NaN,0.0,NaN,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0
11798,475987700,惊喜,a,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,NaN,0.0,NaN,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0
11799,475987700,哭泣,v,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,NaN,0.0,NaN,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0
11800,475987700,儿时,t,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,NaN,0.0,NaN,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0


In [88]:
df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
df_merged_chn

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,album_name,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year
0,641644435,我,r,16,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
1,641644435,清风,n,7,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
2,641644435,天地,n,5,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
3,641644435,伴,v,5,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
4,641644435,醉,v,5,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,清风醉,85081075.0,003yPfqX3jjXlP,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11797,475987700,天赋,n,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,NaN,0.0,NaN,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0
11798,475987700,惊喜,a,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,NaN,0.0,NaN,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0
11799,475987700,哭泣,v,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,NaN,0.0,NaN,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0
11800,475987700,儿时,t,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,NaN,0.0,NaN,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0


In [89]:
df_merged_chn['song_name_pure'].nunique()

120

In [90]:
df_songs_part = df_merged_chn[[
    'song_name_pure'
]].drop_duplicates(keep='first').reset_index(drop=True)
df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                         1).astype(str)
df_songs_part['album_order'] = df_songs_part.index // 10
df_songs_part

,song_name_pure,album_name,album_order
0,清风醉,PART 1,0
1,下个，路口，见,PART 1,0
2,大梦归离,PART 1,0
3,蜀绣,PART 1,0
4,1987我不知会遇见你,PART 1,0
...,...,...,...
115,爱有引力,PART 12,11
116,我的黑白色,PART 12,11
117,TMD我爱你,PART 12,11
118,木兰,PART 12,11


In [91]:
# 虚拟专辑数据，index//12+1作为虚拟专辑
df_merged_chn = df_merged_chn.copy()
df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
df_merged_chn = df_merged_chn.drop(columns=['album_name'])
df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
df_merged_chn

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year,album_name_raw,album_name,album_order
0,641644435,我,r,16,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0,清风醉,PART 1,0
1,641644435,清风,n,7,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0,清风醉,PART 1,0
2,641644435,天地,n,5,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0,清风醉,PART 1,0
3,641644435,伴,v,5,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0,清风醉,PART 1,0
4,641644435,醉,v,5,1,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615.0,...,235.0,1.771466e+09,清风醉,清风醉,清风醉,2026-02-19,2026.0,清风醉,PART 1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10908,475987700,天赋,n,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0,NaN,PART 12,11
10909,475987700,惊喜,a,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0,NaN,PART 12,11
10910,475987700,哭泣,v,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0,NaN,PART 12,11
10911,475987700,儿时,t,1,1,003uThZ34YgICF,今天有朵云爱我 (2024周末愉快演唱会广州站),NaN,李宇春,0.0,...,355.0,1.709914e+09,今天有朵云爱我,NaN,今天有朵云爱我,2024-03-09,2024.0,NaN,PART 12,11


In [92]:
df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

In [19]:
df_merged_chn['pos'].unique()

array(['u', 'y', 't', 'r', 'd', 'v', 'n', 'a', 'c', 'f', 'm', 'q', 'p',
       'an', 'z', 'ad', 'vn', 'Ng', 's', 'l', 'b', 'i', 'ns', 'o', 'nz',
       'nr', 'Vg', 'Ag', 'Tg', 'nt', 'e', 'k', 'vd', 'j', 'Mg', 'Rg'],
      dtype=object)

# 歌曲数据更新

In [79]:
# 虚拟专辑数据，index//12+1作为虚拟专辑
df_songs_final = df_songs.copy()
df_songs_final['album_name_raw'] = df_songs_final['album_name']
df_songs_final = df_songs_final.drop(columns=['album_name'])
df_songs_final['album_name'] = "PART " + (df_songs_final.index // 10 +
                                1).astype(str)
df_songs_final['album_order'] = df_songs_final.index // 10 
df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year,album_name_raw,album_name,album_order
0,641644435,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615,002ZOuVm3Qn20Y,85081075,003yPfqX3jjXlP,235,1771466400,清风醉,清风醉,清风醉,2026-02-19,2026,清风醉,PART 1,0
1,625008,001DIsVq3sSjGV,下个，路口，见,NaN,李宇春,4615,002ZOuVm3Qn20Y,53014,004Z4sId156J79,210,1261411200,下个，路口，见,Chris Lee 同名专辑,下个，路口，见,2009-12-22,2009,Chris Lee 同名专辑,PART 1,0
2,517385504,003SqJTl4fnhRC,大梦归离,《大梦归离》影视剧主题曲,李宇春,4615,002ZOuVm3Qn20Y,57721256,003up2WR1OtPci,281,1727233200,大梦归离,大梦归离 影视原声带,大梦归离,2024-09-25,2024,大梦归离 影视原声带,PART 1,0
3,212992885,001TmXYt4B8W4d,蜀绣 (Live),NaN,李宇春,4615,002ZOuVm3Qn20Y,3899212,003IqTUf2cTV2s,307,1517328000,蜀绣,李宇春野蛮生长巡演LIVE自选,蜀绣,2018-01-31,2018,李宇春野蛮生长巡演LIVE自选,PART 1,0
4,213468025,00467LeA1TSU69,1987我不知会遇见你,NaN,李宇春,4615,002ZOuVm3Qn20Y,3975872,003UqEXI3NvbEl,332,1522512000,1987我不知会遇见你,李宇春2018流行（liú xíng）巡演LIVE辑,1987我不知会遇见你,2018-04-01,2018,李宇春2018流行（liú xíng）巡演LIVE辑,PART 1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,105206640,001oOToq0IUkiM,爱有引力,NaN,李宇春,4615,002ZOuVm3Qn20Y,1235613,000wzySP2ujpo3,221,1449676800,爱有引力,混蛋，我想你,爱有引力,2015-12-10,2015,混蛋，我想你,PART 12,11
119,410614,0033rZ6R11iy99,我的黑白色,NaN,李宇春,4615,002ZOuVm3Qn20Y,33189,001SqLCp2RidK7,301,1193932800,我的黑白色,我的,我的黑白色,2007-11-02,2007,我的,PART 12,11
120,426544,004JYRZc0OGTCi,TMD我爱你,NaN,李宇春,4615,002ZOuVm3Qn20Y,34687,001uSTif44XUoj,200,1210780800,TMD我爱你,超级女声 美梦成真,TMD我爱你,2008-05-15,2008,超级女声 美梦成真,PART 13,12
121,225559362,003SszVH4IlNqh,木兰,《王者荣耀》花木兰英雄主打歌,李宇春,4615,002ZOuVm3Qn20Y,4867924,00436LYE43MyXL,201,1540483200,木兰,天美十年典藏：全明星音乐特辑,木兰,2018-10-26,2018,天美十年典藏：全明星音乐特辑,PART 13,12


In [80]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# 测试